# GW150914: Signal and Noise in LIGO Detector Data

An exploratory analysis of the public 32-second GWOSC data segment for Hanford (H1) and Livingston (L1). What can we observe near the published event time, and what cannot be inferred from a simple comparison?

**Scope:** GW150914 is a known event. This project does not claim a new discovery or a validated detection system; it demonstrates reproducible analysis and visualization of public scientific data.

## Data and processing

The official GWOSC H1 and L1 HDF5 files contain 32 seconds of strain sampled at 4096 Hz. For each detector, the script estimates a Welch PSD from one-second windows with 50% overlap, whitens the strain in the frequency domain, and applies a 20–300 Hz band-pass filter. It compares a 0.20-second event window centered at GPS 1126259462.422 with 298 same-length background windows stepped every 0.10 seconds and centered more than one second away from the event. The required DATA, CBC_CAT1, and CBC_CAT2 flags pass in both files; L1's `NO_CW_HW_INJ` flag is not passed, indicating a continuous-wave hardware injection in this segment.

The `src/analyze_event.py` script downloads the files if needed, repeats the processing, and creates the summary table and figures.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'reports').exists():
    ROOT = ROOT.parent
summary = pd.read_csv(ROOT / 'reports' / 'detector_summary.csv')
summary

## Event versus background

| Detector | Event whitened RMS (σ) | Background median RMS (σ) | Event RMS percentile | Event peak percentile |
|---|---:|---:|---:|---:|
| H1 | 0.704 | 0.340 | 99.3% | 99.3% |
| L1 | 0.564 | 0.342 | 97.7% | 98.3% |

The event-centered RMS and peak are high relative to most same-length windows in this short segment. These empirical percentiles are **not** p-values, false-alarm rates, or detection probabilities: the background windows overlap, the sample is short, and detector noise is nonstationary. L1 also has a continuous-wave hardware-injection flag set, which is a caveat when interpreting its noise spectrum.

![Event-window features compared with background windows](../reports/figures/background_comparison.png)

## Two-detector comparison

L1 is shifted forward by 7 ms and sign-inverted for this reference-based visual alignment, following the educational GWOSC tutorial. This project does not fit the delay.

![Whitened H1 and L1 strain](../reports/figures/detector_coincidence.png)

## Numerical-relativity template comparison

A public educational waveform template is whitened with each detector's PSD, band-passed in the same range, and compared with the data using normalized cross-correlation. The event score is the maximum absolute correlation when the template merger is within ±0.05 seconds of the published trigger.

| Detector | Peak normalized correlation | Best merger offset | Off-event slides below event |
|---|---:|---:|---:|
| H1 | 0.709 | +0.0006 s | 293 / 293 |
| L1 | 0.567 | +0.0004 s | 293 / 293 |

The template match is larger than every sampled off-event slide in this segment. The slides overlap heavily, only 32 seconds are available, and L1 has a continuous-wave injection flag set. This is a model-consistency check for a known event, not a calibrated false-alarm rate or independent detection.

![Template correlation near GW150914](../reports/figures/template_match_near_event.png)

![H1 template overlay](../reports/figures/h1_template_overlay.png)

![L1 template overlay](../reports/figures/l1_template_overlay.png)

The spectrograms show time-frequency power relative to each frequency bin's median level over the full 32-second segment.

![H1 time-frequency view](../reports/figures/h1_spectrogram.png)

![L1 time-frequency view](../reports/figures/l1_spectrogram.png)

## Interpretation and limitations

The whitened event-window statistics rank high against the selected background windows, and the two whitened traces show a similar short transient near the published trigger after a reference-based alignment. This is an exploratory consistency check for a known event. A real search would need longer off-source data, data-quality checks, a full template bank with a calibrated matched-filter pipeline, and a calibrated false-alarm estimate. This project does not claim an independent detection.

## Reproducibility and source

Official sources: the [GWOSC API](https://gwosc.org/api/), [GWTC-1 release for GW150914](https://gwosc.org/eventapi/json/GWTC-1-confident/GW150914/v3/), and the [GWOSC signal-processing tutorial](https://gwosc.org/s/events/GW150914/GW150914_tutorial.html). Data are released under CC BY 4.0; see the [GWOSC acknowledgement and citation guidance](https://gwosc.org/acknowledgements/). Run `python src/analyze_event.py` to download the strain files and waveform template, then reproduce the table and figures.